In [1]:
CAPTIONS_CSV = '/home/jovyan/AA 25-26/FLIR/captions_FLIR_augmented.csv'
YAML_PATH = '/home/jovyan/AA 25-26/FLIR/data.yaml'
MODEL_SAVE_PATH = '/home/jovyan/AA 25-26/ir_caption_model/caption_model.pt'

In [3]:
# ============================================================================
# IR CAPTION MODEL - TRAINING SCRIPT
# Compatible with LLM_and_Image_Encorperated inference notebook
# ============================================================================
#
# USAGE:
#   1. Update the paths in the CONFIG section below
#   2. Run all cells
#   3. The trained model saves to caption_model.pt
#   4. Use your existing inference notebook to generate captions
#
# ============================================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import yaml
import re
from pathlib import Path
from collections import Counter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ============================================================================
# CONFIG - UPDATE THESE PATHS
# ============================================================================

CAPTIONS_CSV = '/home/jovyan/AA 25-26/FLIR/captions_FLIR_augmented.csv'  # <-- your new augmented CSV
YAML_PATH = '/home/jovyan/AA 25-26/FLIR/data.yaml'                       # <-- your YOLO data.yaml
MODEL_SAVE_PATH = '/home/jovyan/AA 25-26/ir_caption_model/caption_model.pt'

# Training hyperparameters
EPOCHS = 150          # increase if loss hasn't plateaued
BATCH_SIZE = 8        # small dataset, small batch
LEARNING_RATE = 0.001
EMBED_DIM = 256
HIDDEN_DIM = 512
NUM_LAYERS = 2
DROPOUT = 0.3         # added dropout to prevent overfitting on 74 samples
MAX_CAPTION_LEN = 50
MIN_WORD_FREQ = 1     # keep ALL words (no frequency cutoff!)
TEACHER_FORCING = 0.8 # probability of using ground truth during training

# ============================================================================
# VOCABULARY - identical to your inference notebook
# ============================================================================

class Vocabulary:
    def __init__(self):
        self.word2idx = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
        self.idx2word = {0: '<PAD>', 1: '<SOS>', 2: '<EOS>', 3: '<UNK>'}
        self.word_count = Counter()
        self.n_words = 4

    def add_word(self, word):
        """Add a word to the vocabulary."""
        if word not in self.word2idx:
            self.word2idx[word] = self.n_words
            self.idx2word[self.n_words] = word
            self.n_words += 1
        self.word_count[word] += 1

    def tokenize(self, sentence):
        sentence = sentence.lower()
        sentence = re.sub(r'[^\w\s\-\+]', ' ', sentence)
        return sentence.split()

    def encode(self, sentence):
        """Convert sentence to list of token indices."""
        tokens = self.tokenize(sentence)
        indices = [self.word2idx['<SOS>']]
        for token in tokens:
            indices.append(self.word2idx.get(token, self.word2idx['<UNK>']))
        indices.append(self.word2idx['<EOS>'])
        return indices

    def decode(self, indices):
        words = []
        for idx in indices:
            if idx == self.word2idx['<EOS>']:
                break
            if idx == self.word2idx['<PAD>'] or idx == self.word2idx['<SOS>']:
                continue
            words.append(self.idx2word.get(idx, '<UNK>'))
        return ' '.join(words)

    def build_from_captions(self, captions, min_freq=1):
        """Build vocabulary from list of caption strings."""
        # First pass: count all words
        for caption in captions:
            tokens = self.tokenize(caption)
            for token in tokens:
                self.word_count[token] += 1

        # Second pass: add words meeting frequency threshold
        for word, count in self.word_count.items():
            if count >= min_freq and word not in self.word2idx:
                self.word2idx[word] = self.n_words
                self.idx2word[self.n_words] = word
                self.n_words += 1

        print(f"  Vocabulary built: {self.n_words} words (min_freq={min_freq})")
        print(f"  Words excluded by frequency: {sum(1 for w,c in self.word_count.items() if c < min_freq)}")

        # Show any words that would become <UNK>
        unk_words = [w for w, c in self.word_count.items() if c < min_freq]
        if unk_words:
            print(f"Excluded words: {unk_words}")
        else:
            print(f" All words included in vocabulary!")


# ============================================================================
# CAPTION MODEL - identical architecture to your inference notebook
# ============================================================================

class CaptionModel(nn.Module):
    def __init__(self, feature_dim, vocab_size, embed_dim=256, hidden_dim=512, num_layers=2, dropout=0.0):
        super(CaptionModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.feature_encoder = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim + hidden_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, features, captions, teacher_forcing_ratio=0.8):
        """
        Training forward pass with teacher forcing.

        Args:
            features: (batch, feature_dim) YOLO detection features
            captions: (batch, seq_len) ground truth token indices
            teacher_forcing_ratio: probability of using ground truth token as next input
        Returns:
            outputs: (batch, seq_len-1, vocab_size) predicted logits
        """
        batch_size = features.size(0)
        seq_len = captions.size(1)

        encoded_features = self.feature_encoder(features)  # (batch, hidden_dim)

        outputs = torch.zeros(batch_size, seq_len - 1, self.output.out_features).to(features.device)
        hidden = None

        # First input is <SOS>
        current_token = captions[:, 0].unsqueeze(1)  # (batch, 1)

        for t in range(1, seq_len):
            embedded = self.embedding(current_token)  # (batch, 1, embed_dim)
            lstm_input = torch.cat([embedded, encoded_features.unsqueeze(1)], dim=2)
            lstm_out, hidden = self.lstm(lstm_input, hidden)
            logits = self.output(lstm_out.squeeze(1))  # (batch, vocab_size)
            outputs[:, t - 1, :] = logits

            # Teacher forcing: use ground truth or model prediction
            if np.random.random() < teacher_forcing_ratio:
                current_token = captions[:, t].unsqueeze(1)
            else:
                current_token = logits.argmax(dim=-1).unsqueeze(1)

        return outputs

    def generate(self, features, vocab, max_len=50, temperature=0.8):
        """Inference generation - identical to your inference notebook."""
        self.eval()
        if features.dim() == 1:
            features = features.unsqueeze(0)

        encoded_features = self.feature_encoder(features)
        current_token = torch.tensor([[vocab.word2idx['<SOS>']]], device=features.device)

        generated_indices = []
        hidden = None

        for _ in range(max_len):
            embedded = self.embedding(current_token)
            lstm_input = torch.cat([embedded, encoded_features.unsqueeze(1)], dim=2)
            lstm_out, hidden = self.lstm(lstm_input, hidden)
            logits = self.output(lstm_out.squeeze(1))
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1)

            if next_token.item() == vocab.word2idx['<EOS>']:
                break

            generated_indices.append(next_token.item())
            current_token = next_token

        return vocab.decode(generated_indices)


# ============================================================================
# YOLO FEATURE EXTRACTOR - identical to your inference notebook
# ============================================================================

class YOLOFeatureExtractor:
    def __init__(self, yaml_path):
        with open(yaml_path, 'r') as f:
            self.config = yaml.safe_load(f)

        self.class_names = self.config['names']
        self.train_path = Path(self.config['train'])
        self.val_path = Path(self.config['val'])

        self.train_label_path = self.train_path.parent.parent / 'train' / 'labels'
        self.val_label_path = self.val_path.parent.parent / 'valid' / 'labels'

    def get_label_path(self, image_path, split='train'):
        label_base = self.train_label_path if split == 'train' else self.val_label_path
        return label_base / f"{Path(image_path).stem}.txt"

    def parse_yolo_label(self, label_path):
        detections = []
        if not Path(label_path).exists():
            return detections
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    detections.append({
                        'class': self.class_names[int(parts[0])],
                        'class_id': int(parts[0]),
                        'x': float(parts[1]),
                        'y': float(parts[2]),
                        'w': float(parts[3]),
                        'h': float(parts[4])
                    })
        return detections

    def extract_features(self, detections, max_objects=20):
        person_count = sum(1 for d in detections if d['class'] == 'person')
        large_vehicle_count = sum(1 for d in detections if d['class'] == 'large vehicle')
        small_vehicle_count = sum(1 for d in detections if d['class'] == 'small vehicle')

        persons = [d for d in detections if d['class'] == 'person']
        vehicles = [d for d in detections if 'vehicle' in d['class']]

        avg_person_x = np.mean([d['x'] for d in persons]) if persons else 0.5
        avg_person_y = np.mean([d['y'] for d in persons]) if persons else 0.5
        avg_vehicle_x = np.mean([d['x'] for d in vehicles]) if vehicles else 0.5
        avg_vehicle_y = np.mean([d['y'] for d in vehicles]) if vehicles else 0.5

        x_spread = np.std([d['x'] for d in detections]) if len(detections) > 1 else 0.0
        y_spread = np.std([d['y'] for d in detections]) if len(detections) > 1 else 0.0

        global_features = [
            min(person_count, 15) / 15.0,
            min(large_vehicle_count, 15) / 15.0,
            min(small_vehicle_count, 15) / 15.0,
            avg_person_x, avg_person_y, avg_vehicle_x, avg_vehicle_y,
            x_spread, y_spread
        ]

        object_features = []
        for i in range(max_objects):
            if i < len(detections):
                d = detections[i]
                class_onehot = [1 if d['class'] == 'person' else 0,
                                1 if d['class'] == 'large vehicle' else 0,
                                1 if d['class'] == 'small vehicle' else 0]
                object_features.extend(class_onehot + [d['x'], d['y'], d['w'], d['h']])
            else:
                object_features.extend([0, 0, 0, 0, 0, 0, 0])

        return np.array(global_features + object_features, dtype=np.float32)


# ============================================================================
# DATASET
# ============================================================================

class CaptionDataset(Dataset):
    def __init__(self, captions_csv, yaml_path, vocab, split='train', max_len=50):
        self.vocab = vocab
        self.max_len = max_len
        self.feature_extractor = YOLOFeatureExtractor(yaml_path)

        # Load captions CSV
        df = pd.read_csv(captions_csv)
        df = df[df['split'] == split].reset_index(drop=True)

        self.samples = []
        skipped = 0

        for _, row in df.iterrows():
            image_path = row['image_path']
            caption = row['caption']

            # Get YOLO label path
            label_path = self.feature_extractor.get_label_path(image_path, split)

            if not label_path.exists():
                skipped += 1
                continue

            # Extract features
            detections = self.feature_extractor.parse_yolo_label(label_path)
            features = self.feature_extractor.extract_features(detections)

            # Encode caption
            indices = self.vocab.encode(caption)

            # Pad or truncate
            if len(indices) < max_len:
                indices = indices + [self.vocab.word2idx['<PAD>']] * (max_len - len(indices))
            else:
                indices = indices[:max_len - 1] + [self.vocab.word2idx['<EOS>']]

            self.samples.append({
                'features': torch.tensor(features, dtype=torch.float32),
                'caption': torch.tensor(indices, dtype=torch.long),
                'image_path': image_path,
                'raw_caption': caption
            })

        print(f"  Dataset loaded: {len(self.samples)} samples ({skipped} skipped - no labels)")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        return sample['features'], sample['caption']


# ============================================================================
# TRAINING
# ============================================================================

def train():
    print("=" * 80)
    print("IR CAPTION MODEL - TRAINING")
    print("=" * 80)

    # --- Step 1: Build Vocabulary ---
    print("\n[1/4] Building vocabulary...")
    df = pd.read_csv(CAPTIONS_CSV)
    all_captions = df['caption'].tolist()

    vocab = Vocabulary()
    vocab.build_from_captions(all_captions, min_freq=MIN_WORD_FREQ)

    # Verify no UNK-prone words
    print(f"\n  Vocabulary check:")
    for caption in all_captions:
        tokens = vocab.tokenize(caption)
        for token in tokens:
            if token not in vocab.word2idx:
                print(f"  ⚠️  '{token}' not in vocabulary!")

    # --- Step 2: Create Dataset ---
    print("\n[2/4] Loading dataset...")
    dataset = CaptionDataset(CAPTIONS_CSV, YAML_PATH, vocab, split='train', max_len=MAX_CAPTION_LEN)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)

    print(f"  Batches per epoch: {len(dataloader)}")

    # --- Step 3: Initialize Model ---
    print("\n[3/4] Initializing model...")
    model = CaptionModel(
        feature_dim=149,
        vocab_size=vocab.n_words,
        embed_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT
    ).to(device)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"  Model parameters: {total_params:,}")
    print(f"  Vocab size: {vocab.n_words}")

    # Loss and optimizer
    criterion = nn.CrossEntropyLoss(ignore_index=vocab.word2idx['<PAD>'])
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=15, verbose=True)

    # --- Step 4: Train ---
    print(f"\n[4/4] Training for {EPOCHS} epochs...")
    print("-" * 60)

    best_loss = float('inf')

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        num_batches = 0

        for features, captions in dataloader:
            features = features.to(device)
            captions = captions.to(device)

            optimizer.zero_grad()

            # Forward pass with teacher forcing
            outputs = model(features, captions, teacher_forcing_ratio=TEACHER_FORCING)

            # Reshape for loss: (batch * seq_len, vocab_size) vs (batch * seq_len)
            outputs = outputs.reshape(-1, vocab.n_words)
            targets = captions[:, 1:].reshape(-1)  # shift by 1 (skip <SOS>)

            loss = criterion(outputs, targets)
            loss.backward()

            # Gradient clipping to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

            optimizer.step()

            total_loss += loss.item()
            num_batches += 1

        avg_loss = total_loss / num_batches
        scheduler.step(avg_loss)

        # Save best model
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save({
                'model_state_dict': model.state_dict(),
                'vocab': vocab,
                'epoch': epoch,
                'loss': best_loss,
            }, MODEL_SAVE_PATH)

        # Print progress
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:3d}/{EPOCHS} | Loss: {avg_loss:.4f} | Best: {best_loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

            # Generate a sample caption to monitor quality
            model.eval()
            with torch.no_grad():
                sample_features = dataset.samples[0]['features'].to(device)
                sample_caption = model.generate(sample_features, vocab, temperature=0.7)
                print(f"          Sample: {sample_caption}")
            model.train()

    print("-" * 60)
    print(f"\n✓ Training complete!")
    print(f"  Best loss: {best_loss:.4f}")
    print(f"  Model saved to: {MODEL_SAVE_PATH}")

    # --- Final: Generate a few sample captions ---
    print(f"\n{'='*80}")
    print("SAMPLE OUTPUTS")
    print("=" * 80)

    model.eval()
    with torch.no_grad():
        for i in range(min(5, len(dataset.samples))):
            features = dataset.samples[i]['features'].to(device)
            raw = dataset.samples[i]['raw_caption']
            generated = model.generate(features, vocab, temperature=0.7)
            print(f"\n  Image: {dataset.samples[i]['image_path'][:50]}...")
            print(f"  Truth: {raw}")
            print(f"  Model: {generated}")

    print(f"\n{'='*80}")
    print("DONE - You can now use your inference notebook with the new model.")
    print("=" * 80)


# ============================================================================
# RUN
# ============================================================================
if __name__ == '__main__':
    train()

Using device: cuda
IR CAPTION MODEL - TRAINING

[1/4] Building vocabulary...
  Vocabulary built: 79 words (min_freq=1)
  Words excluded by frequency: 0
  ✓ All words included in vocabulary!

  Vocabulary check:

[2/4] Loading dataset...
  Dataset loaded: 30 samples (44 skipped - no labels)
  Batches per epoch: 4

[3/4] Initializing model...
  Model parameters: 5,126,991
  Vocab size: 79

[4/4] Training for 150 epochs...
------------------------------------------------------------
  Epoch   1/150 | Loss: 4.2489 | Best: 4.2489 | LR: 0.001000
          Sample: and containing no image coastal parked residential street
  Epoch  10/150 | Loss: 1.6218 | Best: 1.6218 | LR: 0.001000
          Sample: flir image in an urban urban street crossing cars parked and the left standing side of the road
  Epoch  20/150 | Loss: 0.5752 | Best: 0.5195 | LR: 0.001000
          Sample: flir image on a neighborhood 1 person crossing the street street a middle 1 person walking on the right side sidewalk cars p